# Phase 4.1 — main model with slower training (lr 1e-4, dropout 0.2) on Colab GPU

**Before running:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → **Run all** and keep the tab open
(roughly 15–45 minutes; the run stops early when the inner hold-out stops improving).

**One change, fixed in advance (not a search):** the v1 main model stopped at epochs 4 / 2 / 1 with lr 1e-3, so it overfits fast.
This run uses the PatchTST paper defaults, **lr 1e-4 and dropout 0.2** (Nie et al., 2023). To give slower learning room,
max epochs goes from 15 to 30 and early-stopping patience from 3 to 10.
Everything else (model `multiscale_sector` + news, seeds 20261015/16/17, look-back 60, data, splits) is identical to the
v1 run `main_model_v1_news_r2`, so the two compare like for like.

What counts as fixed: best epochs well past 4, and confusion-grid rows that separate by more than v1's +2 points.

Outputs in `MyDrive/Senior/results/`: `main_model_v1_news_p41.json` (scores, grids, settings, per-epoch history),
`main_model_v1_news_p41_preds.csv.gz` (every prediction, per seed) and `results_summary_p41.xlsx` (v1 vs Phase 4.1).
Validation block only; the test block stays sealed.


In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib
for f in ['scripts/baselines_deep.py', 'scripts/summarize_results.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
src = open('scripts/baselines_deep.py', encoding='utf-8').read()
assert '--dropout' in src and 'SET["dropout"]' in src, 'stop: push the latest baselines_deep.py from your PC (with the --lr/--dropout/--patience flags), then run again'


In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
!python tests/test_pipeline.py

In [ ]:
# Phase 4.1: main model WITH news, lr 1e-4, dropout 0.2 (all other settings as in v1)
!python scripts/baselines_deep.py --models multiscale_sector --news data/derived/sentiment_scores.csv \
    --lr 1e-4 --dropout 0.2 --max-epochs 30 --patience 10 \
    --out "$OUT_DIR/main_model_v1_news_p41.json"


In [ ]:
# Did it train longer? Best epoch per seed (v1 was 4 / 2 / 1) and the settings actually used
import json
r = json.load(open(f'{OUT_DIR}/main_model_v1_news_p41.json'))
print({k: r['settings'][k] for k in ('lr', 'dropout', 'max_epochs', 'patience')}, r['git_commit'], r['gpu'], r['torch'])
for run in r['models']['multiscale_sector']['runs']:
    print('seed', run.get('seed'), 'best epoch', run.get('best_epoch'), 'of', len(run.get('history', [])),
          '| val macro-F1', {h: round(run['val'][h]['macro_f1'], 3) for h in ('h1', 'h5', 'h10')})


In [ ]:
# Side-by-side spreadsheet: v1 (lr 1e-3) vs Phase 4.1 (Claude reads this from Drive)
!python scripts/summarize_results.py --inputs "$OUT_DIR/main_model_v1_news_r2.json" "$OUT_DIR/main_model_v1_news_p41.json" --out "$OUT_DIR/results_summary_p41.xlsx"
